# E1 · W8 · Kaggle run 17 — kiểm lại V2 (NMS cùng lớp IoU 0,5) trong detector + chẩn đoán hộp trùng (GPU)
**Mục đích (issue W8-02, ADR-0012 mục 5, `docs/w8/W8_PLAN.md` mục B–C):**
1. Chạy lại đúng job run 14 (YOLO11n `s20261030` + ByteTrack-e1, 38 video **val**), chỉ đổi **một** yếu tố: NMS IoU 0,7 → **0,5** chạy **bên trong** detector. Luật “cùng chiều” đã ghi trước trong W8_PLAN mục B; Claude áp luật ngoài Kaggle.
2. Chẩn đoán vì sao detection video có nhiều hộp trùng hơn dự đoán ảnh val: 4 điều kiện (ảnh JPG / frame giải mã × fp32 / fp16), cùng weights, NMS 0,7.
3. Mã hóa lại 38 video val sang H.264 (giữ số frame, fps, kích thước) để dashboard phát lại W8 (ADR-0015) phát được trên trình duyệt. Không đổi nội dung, chỉ đổi codec.

Test vẫn khóa. Không đổi weights, tracker hay luật cảnh báo.

**Trước khi chạy:** xem hướng dẫn [LOCAL] `docs/w8/kaggle_guide_w8_LOCAL.md`.


In [ ]:
COMMIT = ""                      # <-- SHA day du 40 ky tu (main sau khi merge PR cua W8-02)
TRACKERS = ["tracker_bytetrack_e1.yaml"]
WEIGHTS_SHA = "a3de1e0195e4380aac497612296ed57f67e41ceb5ae3314e82a5ad66b125ef8a"  # best.pt yolo11n_640_s20261030 (run 13)
ULTRALYTICS_VERSION = "8.4.183"  # giu dung ban cua run 12/13/14
NMS_IOU = 0.5                    # V2 (ADR-0012); run 14 dung 0,7
TAG = "w8_run17"


In [ ]:
import glob, hashlib, json, os, sys, zipfile
from pathlib import Path

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"KHONG TIM THAY {name} trong /kaggle/input — kiem tra Add Input"
    if len(hits) > 1:
        print(f"[canh bao] {name} co {len(hits)} ban, dung ban dau: {hits}")
    return hits[0]

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT.lower()), \
    "Dien COMMIT = SHA day du 40 ky tu cua commit tren main chua cac script (GitHub Desktop -> History -> Copy SHA)"

In [ ]:
!pip -q install mcos-decoder "lap>=0.5.12" {"ultralytics==" + ULTRALYTICS_VERSION}
import mcos_decoder, lap, torch, ultralytics
print("ultralytics", ultralytics.__version__, "| lap", lap.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Chua bat GPU: Settings -> Accelerator -> GPU T4 x2"

In [ ]:
# Tim thu muc du lieu tho; neu dataset tho la file .zip thi giai nen vao /kaggle/working/raw
if not glob.glob("/kaggle/input/**/Data/Video_V", recursive=True) and not glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True):
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        if "e1-yolo" in z:
            continue
        print("giai nen", z); zipfile.ZipFile(z).extractall("/kaggle/working/raw")
DATA_ROOT = "/kaggle/working/raw" if glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True) else "/kaggle/input"
assert glob.glob(f"{DATA_ROOT}/**/Data/Video_V", recursive=True), "Khong thay Data/Video_V — kiem tra input e1-svanstrom-raw"
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
SCRIPT = find("w5_track_videos.py")
for n in ["w5_track_metrics.py", "w3_export_yolo.py"]:
    assert os.path.dirname(find(n)) == os.path.dirname(SCRIPT), f"{n} phai nam cung thu muc voi w5_track_videos.py"
MANIFEST = find("yolo_manifest.csv.gz"); AUDIT = find("annotation_audit.csv")
CFGS = [find(t) for t in TRACKERS]
W = [p for p in sorted(glob.glob("/kaggle/input/**/best.pt", recursive=True)) if sha(p) == WEIGHTS_SHA]
assert W, "Khong thay best.pt co SHA-256 cua yolo11n_640_s20261030 — kiem tra input e1-w5-run13"
WEIGHTS = W[0]
for p in (SCRIPT, MANIFEST, AUDIT, WEIGHTS, *CFGS):
    print(f"sha256={sha(p)[:16]}  {p}")

In [ ]:
# Du lieu YOLO e1-yolo-v1 (anh val JPG) cho buoc chan doan; giong run 13
EXPECTED_FP = "d0ac7e230b385132c4bbec90eb5af2c5740a478fec6c79aeb245acca032c27de"
zips = sorted(glob.glob("/kaggle/input/**/e1-yolo-v1.zip", recursive=True))
assert zips, "Khong thay e1-yolo-v1.zip — kiem tra input e1-w3-run03b"
zinfo = json.load(open(find("dataset_zip.json")))
assert sha(zips[0]) == zinfo["sha256"], "SHA-256 cua e1-yolo-v1.zip khac dataset_zip.json"
if not os.path.exists("/tmp/e1data/e1-yolo-v1/data.yaml"):
    zipfile.ZipFile(zips[0]).extractall("/tmp/e1data")
YOLO_DIR = "/tmp/e1data/e1-yolo-v1"
card = json.load(open(f"{YOLO_DIR}/dataset_card.json"))
assert card["content_fingerprint_sha256"] == EXPECTED_FP, card["content_fingerprint_sha256"]
DIAG = find("w8_dup_diagnostic.py"); TRANS = find("w8_transcode_media.py")
assert os.path.dirname(DIAG) == os.path.dirname(SCRIPT) == os.path.dirname(TRANS), "w8_dup_diagnostic.py, w8_transcode_media.py phai nam cung thu muc voi w5_track_videos.py"
R14 = sorted(glob.glob("/kaggle/input/**/w6_run14/dets", recursive=True))
print("dets run 14 (cong tai tao, tuy chon):", R14[0] if R14 else "KHONG CO -> bo qua cong tai tao")


## 1. Track job với NMS IoU 0,5 (giống run 14, chỉ đổi `--iou`)

In [ ]:
!{sys.executable} {SCRIPT} --data-root {DATA_ROOT} --manifest {MANIFEST} --audit {AUDIT} --weights {WEIGHTS} --weights-sha {WEIGHTS_SHA} --trackers {" ".join(CFGS)} --iou {NMS_IOU} --out /kaggle/working --commit {COMMIT} --tag {TAG} --save-detections --save-gt --save-matches


## 2. Chẩn đoán hộp trùng (NMS 0,7, 4 điều kiện)

In [ ]:
!{sys.executable} {DIAG} --data-dir {YOLO_DIR} --manifest {MANIFEST} --data-root {DATA_ROOT} --weights {WEIGHTS} --weights-sha {WEIGHTS_SHA} {"--run14-dets " + R14[0] if R14 else ""} --out /kaggle/working --commit {COMMIT}


## 3. Mã hóa lại 38 video val sang H.264 cho dashboard (CPU, ffmpeg)

In [ ]:
!{sys.executable} {TRANS} --data-root {DATA_ROOT} --manifest {MANIFEST} --out /kaggle/working/w8_media


## Tự kiểm tra (phải in `PASS`)

In [ ]:
import pandas as pd
out = f"/kaggle/working/{TAG}"
st = json.load(open(f"{out}/stage_status.json")); print({k: v["status"] for k, v in st.items()})
inp = json.load(open(f"{out}/inputs.json"))
s = json.load(open(f"{out}/track_summary.json"))["summary"]
print(pd.DataFrame({k: v["all"] for k, v in s.items()}).T[["videos", "gt_frames", "track_recall_pooled", "id_switches",
      "idsw_per_1000_gt_frames", "videos_single_id", "false_tracks"]].to_string())
ok1 = all(v["status"] == "OK" for v in st.values()) and inp["detector"]["nms_iou"] == NMS_IOU and inp["videos"] == 38
dd = json.load(open("/kaggle/working/w8_dupdiag/dup_diagnostic.json")); sd = json.load(open("/kaggle/working/w8_dupdiag/stage_status.json"))
print(pd.DataFrame(dd["conditions"]).T.to_string()); print("pixel diff:", dd["mean_abs_pixel_diff_jpg_vs_frame"], "| repro:", dd["repro_frame_fp32_vs_run14"])
ok2 = all(v["status"] == "OK" for v in sd.values()) and sd["predict"]["images"] == 1202
mm = json.load(open("/kaggle/working/w8_media/media_manifest.json"))
print("media:", mm["videos"], "video H.264, lech:", mm["mismatch"])
ok3 = mm["videos"] == 38 and not mm["mismatch"]
print("PASS" if ok1 and ok2 and ok3 else "FAIL - gui stage_status.json cua hai buoc va media_manifest.json cho Claude")
for z in (f"{TAG}_evidence.zip", f"{TAG}_demo.zip", "w8_dupdiag_evidence.zip", "w8_media.zip"):
    p = f"/kaggle/working/{z}"; print(z, os.path.getsize(p) // 1024, "KB  sha256", sha(p))
